In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
path="/content/drive/MyDrive/Zindi/Zindi IAIO Crop Yield Estimation Challenge/"

In [55]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

# Load data
train = pd.read_csv(path+'Train.csv')
test = pd.read_csv(path+'Test.csv')

# Save ID for later
test_ids = test['ID']

In [56]:
# Transform target
limit = train['Yield'].quantile(0.99)
train['Yield'] = train['Yield'].clip(upper=limit)
y = np.log1p(train['Yield'])
train = train.drop(['Yield', 'ID'], axis=1)
test = test.drop(['ID'], axis=1)

In [57]:
date_cols = ['CropTillageDate', 'RcNursEstDate', 'SeedingSowingTransplanting', 'Harv_date', 'Threshing_date']

for df in [train, test]:
    # Convert to datetime objects temporarily
    for col in date_cols:
        df[col] = pd.to_datetime(df[col])

    # NEW: Calculate Durations (Days)
    # 1. Growing season (Transplanting to Harvest)
    df['Duration_Growth'] = (df['Harv_date'] - df['SeedingSowingTransplanting']).dt.days

    # 2. Preparation time (Tillage to Sowing)
    df['Duration_Prep'] = (df['SeedingSowingTransplanting'] - df['CropTillageDate']).dt.days

    # 3. Post-harvest drying (Harvest to Threshing)
    df['Duration_PostHarvest'] = (df['Threshing_date'] - df['Harv_date']).dt.days

    # Extract standard date features and drop original date columns
    for col in date_cols:
        df[col + '_month'] = df[col].dt.month
        df[col + '_day'] = df[col].dt.dayofyear
        df.drop(col, axis=1, inplace=True)

In [58]:
cat_cols = train.select_dtypes(include=['object']).columns

for col in cat_cols:
    le = LabelEncoder()
    # Combine train and test to ensure all possible words are accounted for
    full_data = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(full_data)

    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

test = test[train.columns]

In [69]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.005,
    'num_leaves': 80,
    'feature_fraction': 0.75,
    'bagging_fraction': 0.75,
    'bagging_freq': 5,
    'min_data_in_leaf': 30,
    'n_jobs': -1,
    'random_state': 42
}

for fold, (train_idx, val_idx) in enumerate(kf.split(train, y)):
    X_tr, X_val = train.iloc[train_idx], train.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=5000,
        valid_sets=[dtrain, dval],
        callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(500)]
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(test) / kf.n_splits

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

In [76]:
actual_yield = np.expm1(y)
predicted_yield = np.expm1(oof_preds)

mse = mean_squared_error(actual_yield, predicted_yield)

rmse = np.sqrt(mse)

print(f"Local CV RMSE: {rmse}")

Local CV RMSE: 183.02471579202972


In [62]:
# Create a dataframe to analyze errors
error_analysis = pd.DataFrame({
    'Actual': np.expm1(y),
    'Predicted': np.expm1(oof_preds),
    'Acre': train['Acre'] # Assuming 'Acre' is in your training data
})

error_analysis['Error'] = np.abs(error_analysis['Actual'] - error_analysis['Predicted'])
error_analysis['Squared_Error'] = error_analysis['Error']**2

# Show the 10 rows that hurt your RMSE the most
print(error_analysis.sort_values(by='Squared_Error', ascending=False).head(10))

      Actual   Predicted      Acre        Error  Squared_Error
3051  2500.0  227.950545  0.156250  2272.049455   5.162209e+06
2922  2506.2  432.239504  0.227273  2073.960496   4.301312e+06
3073  2506.2  466.099308  0.227273  2040.100692   4.162011e+06
521   2506.2  476.065806  0.272727  2030.134194   4.121445e+06
2270  2506.2  484.870263  0.272727  2021.329737   4.085774e+06
1618  2506.2  487.330041  0.250000  2018.869959   4.075836e+06
1101  2506.2  489.797934  0.181818  2016.402066   4.065877e+06
28    2506.2  502.002271  0.227273  2004.197729   4.016809e+06
3614  2506.2  512.731255  0.181818  1993.468745   3.973918e+06
422   2506.2  642.403399  0.312500  1863.796601   3.473738e+06


In [74]:
final_predictions = np.expm1(test_preds)

print("Test Preds Max:", final_predictions.max())
print("Train Yield Max (Clipped):", np.expm1(y).max())

Test Preds Max: 2356.6641384007394
Train Yield Max (Clipped): 2506.199999999998


In [77]:
final_predictions = np.expm1(test_preds)
final_predictions = np.where(final_predictions < 0, 0, final_predictions) # No negative yields

submission = pd.DataFrame({
    'ID': test_ids,
    'Yield': final_predictions
})

file_name = 'submission_final.csv'
submission.to_csv(file_name, index=False)

print(f"Successfully saved {file_name}!")
print(submission.head())

Successfully saved submission_final.csv!
                ID       Yield
0  ID_F9XXEXN2ADR2  594.186300
1  ID_SO3VW2X4QO93  408.335305
2  ID_UKUQ7JM8E894  461.167032
3  ID_QUISMWEZR2H4  307.395047
4  ID_25JGI455VKCZ  544.888511
